# NB24 — Fixed TFT Encoder

**Goal:** Re-train the TFT Encoder from NB19 with corrected training hyperparameters to overcome training instability that caused AUC=0.8483 in NB19 (fold 2 stopped at best_epoch=4 — VSN gating never converged).

## Root Cause of NB19 Failure

| Issue | NB19 | NB24 fix |
|-------|------|----------|
| Learning rate | 5e-4 (too high — VSN softmax weights collapsed) | 1e-4 |
| Early stopping patience | 10 (fold 2 exited at ep 14, best was ep 4) | 25 |
| Max epochs | 80 | 150 |
| Gradient clipping | max_norm=1.0 | max_norm=0.5 |
| LR warmup | None | 5 epochs linear ramp (1e-5 → 1e-4) |
| Cosine T_max | 50 | 100 |

**Architecture is unchanged** — same GRN, VSN, TFTEncoder, W=15, D_MODEL=64. The all-True mask NaN guard was already present in NB19.

## Why TFT Is Worth Fixing

NB19 TFT achieved Spearman ρ=0.549 vs LGBM and ρ=0.639 vs MLP — the lowest in the entire pipeline. No other model has this diversity profile. If it passes the AUC gate (≥ 0.905), a Hybrid-3seed + TFT rank avg projects to LB ~0.946–0.950, a +0.010–0.015 gain over current best 0.93651.

## Inputs / Outputs
**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`,
`oof_predictions_nb21.parquet` (hybrid_oof), `oof_predictions_nb12.parquet` (lgbm_tuned_oof)

**Outputs:** `oof_predictions_nb24.parquet` (tft24_oof), `test_predictions_nb24.parquet` (tft24_pred),
`models/tft24_fold{0-4}.pt`, `models/nb24_summary.pkl`

## Gate (revised for Phase 4)
- Solo OOF AUC ≥ **0.905**
- Spearman ρ < **0.85** vs LGBM (revised gate — primary diversity measure is vs GBM anchor)

In [ ]:
# nb24-01  Imports
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
import pickle, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# nb24-02  Path detection + data load
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root = cwd
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(exist_ok=True)
print(f'Root: {project_root}')

train    = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test     = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
hybrid_oof = pd.read_parquet(processed_dir / 'oof_predictions_nb21.parquet')[['id', 'hybrid_oof']]
lgbm_oof   = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')[['id', 'lgbm_tuned_oof']]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Folds: {folds_df.fold.value_counts().sort_index().to_dict()}')

In [ ]:
# nb24-03  Settings
# --- NB24 KEY CHANGES vs NB19 ---
# LR_MAX   : 5e-4 → 1e-4   (5x lower; VSN softmax needs slow early updates)
# PATIENCE : 10   → 25     (fold 2 best was ep 4 with 10-epoch patience)
# MAX_EPOCHS: 80  → 150    (room to converge after warmup)
# CLIP     : 1.0  → 0.5    (tighter gradient clipping for VSN stability)
# WARMUP   : 0    → 5 ep   (linear ramp from LR_MAX/10 to LR_MAX)
# T_MAX    : 50   → 100    (cosine annealing covers full training budget)

FEAT_COLS = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq', 'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session', 'Stint', 'Position',
    'Degradation_rate', 'Degradation_acceleration', 'Cumulative_Degradation_winsorized',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3', 'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    'prime_pit_window', 'prime_window_x_compound',
    'abs_position_change', 'pos_change_rolling_std_3', 'PitStop_lag1',
    'laps_to_driver_end', 'field_median_laptime', 'laptime_vs_field', 'field_pace_change',
]
assert len(FEAT_COLS) == 38, f'Expected 38, got {len(FEAT_COLS)}'

SEQ_COLS = ['LapTime (s)', 'TyreLife', 'Cumulative_Degradation_winsorized',
            'LapTime_Delta', 'Position', 'PitStop']
CAT_COLS = ['Driver', 'Race', 'Compound', 'Year']

WINDOW      = 15       # unchanged — attention compensates for shorter raw history
D_MODEL     = 64       # unchanged
NUM_HEADS   = 4        # unchanged
NUM_LAYERS  = 2        # unchanged
DROPOUT     = 0.1      # unchanged
POS_WEIGHT  = 4.03     # unchanged
BATCH_SIZE  = 2048     # unchanged
MAX_EPOCHS  = 150      # NB19=80
PATIENCE    = 25       # NB19=10
LR_MAX      = 1e-4     # NB19=5e-4
WEIGHT_DECAY= 1e-4     # unchanged
CLIP_GRAD   = 0.5      # NB19=1.0
WARMUP_EPOCHS = 5      # NB19=0  (linear ramp from LR_MAX/10 to LR_MAX)
T_MAX       = 100      # NB19=50 (cosine annealing period after warmup)
N_FOLDS     = 5

print(f'Static features: {len(FEAT_COLS)}, Seq features: {len(SEQ_COLS)}, Window: {WINDOW}, d_model: {D_MODEL}')
print(f'LR_MAX={LR_MAX}, PATIENCE={PATIENCE}, MAX_EPOCHS={MAX_EPOCHS}, CLIP_GRAD={CLIP_GRAD}, WARMUP={WARMUP_EPOCHS}')
if DEVICE.type == 'cpu':
    print('WARNING: CPU-only. Upload to Kaggle T4 for ~35-50 min/fold training.')

In [ ]:
# nb24-04  Label encoders for entity embeddings (fit on combined train+test)
combined_cats = pd.concat([train[CAT_COLS], test[CAT_COLS]], ignore_index=True)
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(combined_cats[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} classes')

for col in CAT_COLS:
    le = label_encoders[col]
    train[col + '_idx'] = le.transform(train[col].astype(str))
    test[col  + '_idx'] = le.transform(test[col].astype(str))

EMB_DIMS = {
    'Driver':   (len(label_encoders['Driver'].classes_)   + 1, 32),
    'Race':     (len(label_encoders['Race'].classes_)     + 1,  8),
    'Compound': (len(label_encoders['Compound'].classes_) + 1,  3),
    'Year':     (len(label_encoders['Year'].classes_)     + 1,  2),
}
print('Embedding dims:', EMB_DIMS)

In [ ]:
# nb24-05  Build 15-lap sequence windows  (identical to NB19)
idx_cols     = ['id', 'Race', 'Year', 'Driver', 'LapNumber']
cat_idx_cols = [c + '_idx' for c in CAT_COLS]

seq_scaler = StandardScaler()
seq_scaler.fit(train[SEQ_COLS].values)

all_sel = list(dict.fromkeys(idx_cols + SEQ_COLS + FEAT_COLS + cat_idx_cols))

combined_df = pd.concat([
    train[all_sel + ['PitNextLap']].assign(is_train=True),
    test[ all_sel].assign(is_train=False, PitNextLap=-1),
], ignore_index=True)

combined_df = combined_df.sort_values(['Race', 'Year', 'Driver', 'LapNumber']).reset_index(drop=True)
combined_df[SEQ_COLS] = seq_scaler.transform(combined_df[SEQ_COLS].values).astype(np.float32)

N        = len(combined_df)
N_SF     = len(SEQ_COLS)
seq_vals = combined_df[SEQ_COLS].values.astype(np.float32)

all_windows = np.zeros((N, WINDOW, N_SF), dtype=np.float32)
all_masks   = np.zeros((N, WINDOW),       dtype=bool)

GROUP_COLS = ['Race', 'Year', 'Driver']
print(f'Building {WINDOW}-lap windows for {N:,} rows...')
t0 = time.time()

for _, grp_idx in tqdm(combined_df.groupby(GROUP_COLS, sort=False).groups.items(),
                       total=combined_df[GROUP_COLS].drop_duplicates().shape[0]):
    idxs  = grp_idx.values
    n_grp = len(idxs)
    for i in range(n_grp):
        hist_len = min(i, WINDOW)
        if hist_len > 0:
            all_windows[idxs[i], WINDOW - hist_len:] = seq_vals[idxs[i - hist_len : i]]
            all_masks[idxs[i],   WINDOW - hist_len:] = True

print(f'Done in {time.time()-t0:.1f}s')
print(f'Rows with full history (>=15 prior laps): {all_masks.all(axis=1).sum():,}')
print(f'Rows with zero history: {(~all_masks.any(axis=1)).sum():,}')

In [ ]:
# nb24-06  Split windows back to train / test arrays  (identical to NB19)
tr_mask = combined_df['is_train'].values
te_mask = ~tr_mask

train_windows   = all_windows[tr_mask]
train_seq_masks = all_masks[tr_mask]
test_windows    = all_windows[te_mask]
test_seq_masks  = all_masks[te_mask]

train_static_raw = combined_df.loc[tr_mask, FEAT_COLS].values.astype(np.float32)
test_static_raw  = combined_df.loc[te_mask, FEAT_COLS].values.astype(np.float32)
train_targets    = combined_df.loc[tr_mask, 'PitNextLap'].values.astype(np.float32)
train_ids        = combined_df.loc[tr_mask, 'id'].values
test_ids         = combined_df.loc[te_mask, 'id'].values

fold_lookup = folds_df.set_index('id')['fold']
train_folds = fold_lookup.loc[train_ids].values

train_cat = {c: combined_df.loc[tr_mask, c+'_idx'].values for c in CAT_COLS}
test_cat  = {c: combined_df.loc[te_mask, c+'_idx'].values for c in CAT_COLS}

del all_windows, all_masks, combined_df

print(f'Train windows: {train_windows.shape},  static: {train_static_raw.shape}')
print(f'Test  windows: {test_windows.shape},  static: {test_static_raw.shape}')
print(f'Fold counts: {pd.Series(train_folds).value_counts().sort_index().to_dict()}')

In [ ]:
# nb24-07  GRN + VSN + TFT Encoder model  (IDENTICAL to NB19 — architecture unchanged)

class GRN(nn.Module):
    """Gated Residual Network: LayerNorm -> ELU -> GLU -> residual + LayerNorm."""
    def __init__(self, d_in, d_model, d_context=0, dropout=0.05):
        super().__init__()
        self.skip   = nn.Linear(d_in, d_model) if d_in != d_model else nn.Identity()
        self.ln_in  = nn.LayerNorm(d_in)
        self.fc1    = nn.Linear(d_in + d_context, d_model)
        self.fc2    = nn.Linear(d_model, d_model * 2)
        self.ln_out = nn.LayerNorm(d_model)
        self.drop   = nn.Dropout(dropout)

    def forward(self, a, c=None):
        skip = self.skip(a)
        x    = self.ln_in(a)
        if c is not None:
            if c.dim() < x.dim():
                c = c.unsqueeze(1).expand(*x.shape[:-1], c.shape[-1])
            x = torch.cat([x, c], dim=-1)
        x    = self.drop(F.elu(self.fc1(x)))
        v, g = self.fc2(x).chunk(2, dim=-1)
        return self.ln_out(skip + v * torch.sigmoid(g))


class VSN(nn.Module):
    """Variable Selection Network: learns softmax weights over per-feature projections."""
    def __init__(self, n_feats, d_model, d_context=0, dropout=0.05):
        super().__init__()
        self.projs  = nn.ModuleList([nn.Linear(1, d_model) for _ in range(n_feats)])
        self.w_grn  = GRN(n_feats * d_model, d_model, d_context=d_context, dropout=dropout)
        self.w_fc   = nn.Linear(d_model, n_feats)
        self.ln_out = nn.LayerNorm(d_model)
        self.n, self.d = n_feats, d_model

    def forward(self, x, context=None):
        parts   = [p(x[..., i:i+1]) for i, p in enumerate(self.projs)]
        stacked = torch.stack(parts, dim=-2)                           # (..., n, d)
        flat    = stacked.view(*stacked.shape[:-2], self.n * self.d)   # (..., n*d)
        w       = F.softmax(self.w_fc(self.w_grn(flat, context)), dim=-1)  # (..., n)
        return self.ln_out((stacked * w.unsqueeze(-1)).sum(dim=-2))    # (..., d)


class TFTEncoder(nn.Module):
    """Simplified TFT Encoder: VSN feature selection + static-init LSTM + temporal self-attention."""
    def __init__(self, n_static=38, n_seq=6, d_model=64, num_heads=4,
                 num_lstm_layers=2, dropout=0.1, emb_dims=None):
        super().__init__()
        if emb_dims is None:
            emb_dims = EMB_DIMS
        emb_total = sum(v[1] for v in emb_dims.values())  # 45

        self.driver_emb   = nn.Embedding(*emb_dims['Driver'])
        self.race_emb     = nn.Embedding(*emb_dims['Race'])
        self.compound_emb = nn.Embedding(*emb_dims['Compound'])
        self.year_emb     = nn.Embedding(*emb_dims['Year'])
        self.emb_proj     = nn.Linear(emb_total, d_model)

        self.static_vsn   = VSN(n_static, d_model, d_context=d_model, dropout=dropout)
        self.static_merge = GRN(d_model * 2, d_model, dropout=dropout)

        self.h0_net = nn.Linear(d_model, num_lstm_layers * d_model)
        self.c0_net = nn.Linear(d_model, num_lstm_layers * d_model)

        self.temporal_vsn = VSN(n_seq, d_model, d_context=d_model, dropout=dropout)

        self.lstm = nn.LSTM(d_model, d_model, num_lstm_layers, batch_first=True,
                            dropout=dropout if num_lstm_layers > 1 else 0.0)

        self.attn     = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.attn_ln  = nn.LayerNorm(d_model)
        self.attn_grn = GRN(d_model, d_model, dropout=dropout)

        self.out_grn = GRN(d_model, d_model, dropout=dropout)
        self.head    = nn.Linear(d_model, 1)

        self.d = d_model
        self.L = num_lstm_layers

    def forward(self, window, mask, static, driver, race, compound, year):
        B, T, _ = window.shape

        emb = torch.cat([self.driver_emb(driver), self.race_emb(race),
                         self.compound_emb(compound), self.year_emb(year)], dim=1)
        entity_ctx = F.relu(self.emb_proj(emb))                          # (B, d)

        static_vsn  = self.static_vsn(static, context=entity_ctx)        # (B, d)
        static_repr = self.static_merge(
            torch.cat([static_vsn, entity_ctx], dim=-1))                  # (B, d)

        h0 = self.h0_net(static_repr).view(B, self.L, self.d).permute(1, 0, 2).contiguous()
        c0 = self.c0_net(static_repr).view(B, self.L, self.d).permute(1, 0, 2).contiguous()

        tvn = self.temporal_vsn(window, context=static_repr)             # (B, T, d)

        seq_len = mask.sum(1).long().clamp(min=1)
        packed  = nn.utils.rnn.pack_padded_sequence(
            tvn, seq_len.cpu(), batch_first=True, enforce_sorted=False)
        lstm_out, _ = self.lstm(packed, (h0, c0))
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            lstm_out, batch_first=True, total_length=T)                  # (B, T, d)

        # Guard: rows with all-False mask would produce all-True key_padding_mask
        # -> softmax(-inf) = NaN. Unlock those rows so attention runs unmasked.
        attn_key_mask = ~mask                                             # True = ignore
        no_valid = attn_key_mask.all(dim=1)
        if no_valid.any():
            attn_key_mask = attn_key_mask.clone()
            attn_key_mask[no_valid] = False

        attn_out, _ = self.attn(lstm_out, lstm_out, lstm_out,
                                key_padding_mask=attn_key_mask)
        attn_out    = self.attn_grn(self.attn_ln(lstm_out + attn_out))   # (B, T, d)

        valid  = mask.float().unsqueeze(-1)                              # (B, T, 1)
        pooled = (attn_out * valid).sum(1) / valid.sum(1).clamp(min=1)  # (B, d)

        return self.head(self.out_grn(pooled)).squeeze(1)               # (B,)


_m = TFTEncoder(n_static=38, n_seq=len(SEQ_COLS), d_model=D_MODEL,
                num_heads=NUM_HEADS, num_lstm_layers=NUM_LAYERS, dropout=DROPOUT)
n_params = sum(p.numel() for p in _m.parameters())
print(f'TFT Encoder params: {n_params:,}')
del _m

In [ ]:
# nb24-08  Dataset + DataLoader  (identical to NB19)
class F1SeqDataset(Dataset):
    def __init__(self, windows, masks, static, cat_idxs, targets=None):
        self.windows  = torch.tensor(windows, dtype=torch.float32)
        self.masks    = torch.tensor(masks,   dtype=torch.bool)
        self.static   = torch.tensor(static,  dtype=torch.float32)
        self.driver   = torch.tensor(cat_idxs['Driver'],   dtype=torch.long)
        self.race     = torch.tensor(cat_idxs['Race'],     dtype=torch.long)
        self.compound = torch.tensor(cat_idxs['Compound'], dtype=torch.long)
        self.year     = torch.tensor(cat_idxs['Year'],     dtype=torch.long)
        self.targets  = (None if targets is None
                         else torch.tensor(targets, dtype=torch.float32))

    def __len__(self):  return len(self.windows)

    def __getitem__(self, i):
        d = dict(window=self.windows[i], mask=self.masks[i], static=self.static[i],
                 driver=self.driver[i], race=self.race[i],
                 compound=self.compound[i], year=self.year[i])
        if self.targets is not None:
            d['target'] = self.targets[i]
        return d


def make_loader(windows, masks, static, cat_idxs, targets=None, shuffle=False):
    ds = F1SeqDataset(windows, masks, static, cat_idxs, targets)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

In [ ]:
# nb24-09  Training utilities
# CHANGE vs NB19: gradient clipping norm lowered from 1.0 → CLIP_GRAD=0.5

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_logits = 0.0, []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            win  = batch['window'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            stat = batch['static'].to(DEVICE)
            drv  = batch['driver'].to(DEVICE)
            rc   = batch['race'].to(DEVICE)
            cmp  = batch['compound'].to(DEVICE)
            yr   = batch['year'].to(DEVICE)

            logits = model(win, mask, stat, drv, rc, cmp, yr)

            if is_train:
                tgt  = batch['target'].to(DEVICE)
                loss = criterion(logits, tgt)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)  # 0.5 vs NB19's 1.0
                optimizer.step()
                total_loss += loss.item() * len(win)

            all_logits.append(logits.detach().cpu().float().numpy())

    probs = torch.sigmoid(torch.tensor(np.concatenate(all_logits))).numpy()
    return probs, total_loss / max(len(loader.dataset), 1)


def make_scheduler(optimizer):
    """Linear warmup for WARMUP_EPOCHS then cosine anneal over T_MAX epochs."""
    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=T_MAX, eta_min=LR_MAX * 1e-3)
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

In [ ]:
# nb24-10  5-fold CV training
# KEY CHANGES vs NB19:
#   - LR_MAX=1e-4 (was 5e-4)
#   - PATIENCE=25 (was 10)
#   - MAX_EPOCHS=150 (was 80)
#   - SequentialLR warmup+cosine (was bare CosineAnnealingLR)
#   - CLIP_GRAD=0.5 (applied in run_epoch, was 1.0)

oof_preds        = np.zeros(len(train_ids))
test_preds_folds = []
fold_aucs        = []
best_epochs      = []

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE)
)

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'Fold {fold}')
    print('='*60)

    tr_idx = np.where(train_folds != fold)[0]
    va_idx = np.where(train_folds == fold)[0]

    scaler  = StandardScaler()
    tr_stat = scaler.fit_transform(train_static_raw[tr_idx])
    va_stat = scaler.transform(train_static_raw[va_idx])
    te_stat = scaler.transform(test_static_raw)

    tr_cat = {c: train_cat[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: train_cat[c][va_idx] for c in CAT_COLS}

    tr_loader = make_loader(train_windows[tr_idx], train_seq_masks[tr_idx],
                            tr_stat, tr_cat, targets=train_targets[tr_idx], shuffle=True)
    va_loader = make_loader(train_windows[va_idx], train_seq_masks[va_idx], va_stat, va_cat)
    te_loader = make_loader(test_windows, test_seq_masks, te_stat, test_cat)

    model     = TFTEncoder(n_static=38, n_seq=len(SEQ_COLS), d_model=D_MODEL,
                           num_heads=NUM_HEADS, num_lstm_layers=NUM_LAYERS,
                           dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
    scheduler = make_scheduler(optimizer)

    best_auc, best_epoch, patience_ctr, best_state = 0.0, 0, 0, None

    for epoch in range(MAX_EPOCHS):
        _, tr_loss   = run_epoch(model, tr_loader, criterion, optimizer)
        scheduler.step()
        val_probs, _ = run_epoch(model, va_loader, criterion)
        val_auc      = roc_auc_score(train_targets[va_idx], val_probs)

        if val_auc > best_auc:
            best_auc, best_epoch, patience_ctr = val_auc, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0 or patience_ctr >= PATIENCE:
            current_lr = optimizer.param_groups[0]['lr']
            print(f'  ep {epoch+1:3d}  lr={current_lr:.2e}  loss={tr_loss:.4f}  '
                  f'val={val_auc:.4f}  best={best_auc:.4f} (ep{best_epoch+1})')

        if patience_ctr >= PATIENCE:
            print('  Early stop')
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    oof_probs, _ = run_epoch(model, va_loader, criterion)
    oof_preds[va_idx] = oof_probs
    fold_auc = roc_auc_score(train_targets[va_idx], oof_probs)
    fold_aucs.append(fold_auc)
    best_epochs.append(best_epoch + 1)
    print(f'  Fold {fold} AUC: {fold_auc:.4f}  (best ep {best_epoch+1})')

    te_probs, _ = run_epoch(model, te_loader, criterion)
    test_preds_folds.append(te_probs)

    torch.save(best_state, models_dir / f'tft24_fold{fold}.pt')
    print(f'  Saved tft24_fold{fold}.pt')

oof_auc = roc_auc_score(train_targets, oof_preds)
print(f'\nOOF AUC : {oof_auc:.4f}')
print(f'Fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Fold std : {np.std(fold_aucs):.4f}')
print(f'Best eps : {best_epochs}')

In [ ]:
# nb24-11  Correlation analysis + gate check vs Hybrid NB21 + LGBM
hybrid_vals = hybrid_oof.set_index('id').loc[train_ids, 'hybrid_oof'].values
lgbm_vals   = lgbm_oof.set_index('id').loc[train_ids, 'lgbm_tuned_oof'].values

rho_hybrid, _ = spearmanr(oof_preds, hybrid_vals)
rho_lgbm,   _ = spearmanr(oof_preds, lgbm_vals)
rho_h_lgbm, _ = spearmanr(hybrid_vals, lgbm_vals)

hybrid_auc = roc_auc_score(train_targets, hybrid_vals)
lgbm_auc   = roc_auc_score(train_targets, lgbm_vals)

print(f'TFT24  solo OOF AUC : {oof_auc:.4f}')
print(f'Hybrid solo OOF AUC : {hybrid_auc:.4f}  (NB21 single seed)')
print(f'LGBM   solo OOF AUC : {lgbm_auc:.4f}')
print()
print('Spearman rho:')
print(f'  TFT24 x Hybrid : {rho_hybrid:.4f}')
print(f'  TFT24 x LGBM   : {rho_lgbm:.4f}   (NB19 was 0.549 — expect similar)')
print(f'  Hybrid x LGBM  : {rho_h_lgbm:.4f}')
print()
print(f'Gate: solo AUC >= 0.905  -> {"PASS" if oof_auc >= 0.905 else "FAIL"}')
print(f'Gate: rho  <  0.85 vs LGBM -> {"PASS" if rho_lgbm < 0.85 else "FAIL"}')

# Rank-average ensemble previews
def rank_norm(a):
    return rankdata(a) / len(a)

r_tft    = rank_norm(oof_preds)
r_hybrid = rank_norm(hybrid_vals)
r_lgbm   = rank_norm(lgbm_vals)

auc_tft_hybrid      = roc_auc_score(train_targets, (r_tft + r_hybrid) / 2)
auc_tft_lgbm        = roc_auc_score(train_targets, (r_tft + r_lgbm) / 2)
auc_tft_hybrid_lgbm = roc_auc_score(train_targets, (r_tft + r_hybrid + r_lgbm) / 3)

print(f'\nRank avg TFT24+Hybrid:        {auc_tft_hybrid:.4f}  (delta vs Hybrid: {auc_tft_hybrid - hybrid_auc:+.4f})')
print(f'Rank avg TFT24+LGBM:          {auc_tft_lgbm:.4f}  (delta vs LGBM:   {auc_tft_lgbm - lgbm_auc:+.4f})')
print(f'Rank avg TFT24+Hybrid+LGBM:   {auc_tft_hybrid_lgbm:.4f}')
print()
print('NOTE: Hybrid-3seed rank avg (v010) = 0.9223. Load NB21 3-seed OOF from')
print('      oof_predictions_nb21_seed*.parquet if available, or use single-seed as proxy.')

In [ ]:
# nb24-12  Average test predictions across folds
test_preds_mean = np.mean(np.stack(test_preds_folds, axis=0), axis=0)
print(f'Test preds: shape={test_preds_mean.shape}, mean={test_preds_mean.mean():.4f}, '
      f'std={test_preds_mean.std():.4f}')

In [ ]:
# nb24-13  Save outputs
oof_df = pd.DataFrame({
    'id':         train_ids,
    'fold':       train_folds,
    'PitNextLap': train_targets.astype(int),
    'tft24_oof':  oof_preds,
})
assert len(oof_df) == 439140, f'Expected 439140 rows, got {len(oof_df)}'
assert oof_df['id'].nunique() == 439140, 'Duplicate ids in OOF'
oof_df.to_parquet(processed_dir / 'oof_predictions_nb24.parquet', index=False)
print(f'Saved oof_predictions_nb24.parquet  ({len(oof_df):,} rows)')

test_out = pd.DataFrame({'id': test_ids, 'tft24_pred': test_preds_mean})
test_out.to_parquet(processed_dir / 'test_predictions_nb24.parquet', index=False)
print(f'Saved test_predictions_nb24.parquet ({len(test_out):,} rows)')

summary = {
    'oof_auc':            oof_auc,
    'fold_aucs':          fold_aucs,
    'fold_std':           float(np.std(fold_aucs)),
    'best_epochs':        best_epochs,
    'rho_vs_hybrid':      float(rho_hybrid),
    'rho_vs_lgbm':        float(rho_lgbm),
    'auc_tft_hybrid':     float(auc_tft_hybrid),
    'auc_tft_hybrid_lgbm': float(auc_tft_hybrid_lgbm),
    'model':              'TFT-Encoder-NB24-Fixed',
    'architecture':       f'VSN({len(FEAT_COLS)}+{len(SEQ_COLS)})+StaticInit-LSTM({D_MODEL}x{NUM_LAYERS})+Attn({NUM_HEADS}h)',
    'window_size':        WINDOW,
    'd_model':            D_MODEL,
    'lr_max':             LR_MAX,
    'patience':           PATIENCE,
    'warmup_epochs':      WARMUP_EPOCHS,
    'clip_grad':          CLIP_GRAD,
    'gate_pass':          bool(oof_auc >= 0.905 and rho_lgbm < 0.85),
}
with open(models_dir / 'nb24_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)

print(f'\n=== NB24 Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
# nb24-14  Submission (only if gate passes)
if oof_auc >= 0.905:
    # Load Hybrid 3-seed test preds (v010, the current best)
    # Expected file: test_predictions_nb21.parquet (hybrid_pred) — single seed as proxy
    # For true 3-seed avg, average test_predictions_nb21_seed{0,1,2}.parquet if saved separately
    try:
        hybrid_test = pd.read_parquet(processed_dir / 'test_predictions_nb21.parquet')
        hybrid_test_vals = hybrid_test.set_index('id').loc[test_ids, 'hybrid_pred'].values
        r_tft_test    = rankdata(test_preds_mean) / len(test_preds_mean)
        r_hybrid_test = rankdata(hybrid_test_vals) / len(hybrid_test_vals)
        combined_test = (r_tft_test + r_hybrid_test) / 2
        sub = pd.DataFrame({'id': test_ids, 'PitNextLap': combined_test})
        sub_path = project_root / 'submissions' / 'submission_v012_tft24_hybrid_rankavg.csv'
        sub.to_csv(sub_path, index=False)
        print(f'Submission saved: {sub_path}')
        print(f'CV OOF rank avg TFT24+Hybrid: {auc_tft_hybrid:.4f}')
        print(f'Projected LB (using +0.0166 boost): {auc_tft_hybrid + 0.0166:.4f}')
    except FileNotFoundError as e:
        print(f'Hybrid test file not found: {e}')
        print('Save TFT24 predictions only:')
        sub = pd.DataFrame({'id': test_ids, 'PitNextLap': test_preds_mean})
        sub_path = project_root / 'submissions' / 'submission_v012_tft24_solo.csv'
        sub.to_csv(sub_path, index=False)
        print(f'Solo TFT24 submission: {sub_path}')
else:
    print(f'Gate FAILED (AUC={oof_auc:.4f} < 0.905). Do NOT submit.')
    print('Troubleshooting tips:')
    print(f'  best_epochs per fold: {best_epochs}')
    if min(best_epochs) <= 10:
        print('  -> At least one fold converged very early. Try LR=5e-5 or patience=35.')
    print(f'  fold_aucs: {[f"{a:.4f}" for a in fold_aucs]}')
    low_folds = [i for i, a in enumerate(fold_aucs) if a < 0.87]
    if low_folds:
        print(f'  -> Folds {low_folds} are still failing. Check those folds\' best_epoch values.')